# Cenário: E-commerce (Região Sul)
Este projeto simula um ambiente de E-commerce. Utilizamos duas bases de dados: **Clientes** e **Vendas**. O objetivo é cruzar estas tabelas, filtrar os clientes da região Sul e gravar os dados consolidados.

### Modelo de Entidade-Relacionamento (ER)
O modelo consiste numa relação de **1 para N** (1:N), onde um Cliente pode ter várias Vendas associadas a ele através do `id_cliente`.

![Modelo ER](docs/modelo_er.jpeg)

### Códigos DDL (Estrutura das Tabelas)
Abaixo está a representação da estrutura das nossas tabelas base:

```sql
-- Tabela de Clientes
CREATE TABLE clientes (
    id_cliente INT,
    nome STRING,
    estado STRING
);

-- Tabela de Vendas
CREATE TABLE vendas (
    id_venda INT,
    id_cliente INT,
    data_venda DATE,
    valor FLOAT,
    status STRING
);

# Delta Lake com PySpark
Demonstração de operações CRUD em tabelas Delta Lake usando dados de E-commerce.

In [1]:
import pyspark
from delta import *
from commons import get_raw_data, filter_southern_region

builder = pyspark.sql.SparkSession.builder.appName("DeltaLab") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

spark = configure_spark_with_delta_pip(builder).getOrCreate()
print('Spark iniciado com sucesso!')

26/04/28 20:19:59 WARN Utils: Your hostname, NOTE-LAURA resolves to a loopback address: 127.0.1.1; using 172.17.40.94 instead (on interface eth0)
26/04/28 20:19:59 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/lauras/.cache/pypoetry/virtualenvs/spark-delta-iceberg-Qz15G4zY-py3.12/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/lauras/.ivy2/cache
The jars for the packages stored in: /home/lauras/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-e923af28-da41-4fbc-9d61-44bf1e874294;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 279ms :: artifacts dl 8ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0 

Spark iniciado com sucesso!


## Carregando os dados

In [6]:
df_vendas = get_raw_data(spark, "vendas.csv")
df_clientes = get_raw_data(spark, "clientes.csv")
df_vendas.show(20)
df_clientes.show(20)

+--------+----------+----------+------+-----------+
|id_venda|id_cliente|data_venda| valor|     status|
+--------+----------+----------+------+-----------+
|       1|       101|2023-01-15| 250.5|  concluido|
|       2|       102|2023-01-16| 120.0|  concluido|
|       3|       103|2023-01-17| 340.9|  cancelado|
|       4|       101|2023-01-18|  50.0|  concluido|
|       5|       104|2023-01-19|1500.0|  concluido|
|       6|       105|2023-01-20|  89.9|  concluido|
|       7|       106|2023-01-21| 450.0|   pendente|
|       8|       107|2023-01-22|1200.5|  concluido|
|       9|       108|2023-01-23|299.99|  concluido|
|      10|       109|2023-01-24|  75.0|  cancelado|
|      11|       110|2023-01-25| 600.0|  concluido|
|      12|       111|2023-01-26| 315.2|  concluido|
|      13|       112|2023-01-27| 140.0|processando|
|      14|       113|2023-01-28| 980.0|  concluido|
|      15|       114|2023-01-29|  55.5|  concluido|
|      16|       104|2023-01-30| 210.0|  concluido|
+--------+--

## INSERT — Salvando dados no formato Delta Lake

In [7]:
df_sul = filter_southern_region(df_vendas.join(df_clientes, "id_cliente"))
df_sul.write.format("delta").mode("overwrite").save("../data/delta/vendas_sul")
print('Dados salvos no Delta Lake!')
spark.read.format("delta").load("../data/delta/vendas_sul").show()

Dados salvos no Delta Lake!
+----------+--------+----------+------+---------+--------------+------+
|id_cliente|id_venda|data_venda| valor|   status|          nome|estado|
+----------+--------+----------+------+---------+--------------+------+
|       101|       1|2023-01-15| 250.5|concluido|     Ana Julia|    SC|
|       102|       2|2023-01-16| 120.0|concluido|Laura Silveira|    RS|
|       101|       4|2023-01-18|  50.0|concluido|     Ana Julia|    SC|
|       104|       5|2023-01-19|1500.0|concluido|       Janaina|    PR|
|       110|      11|2023-01-25| 600.0|concluido|  Camila Rocha|    SC|
|       111|      12|2023-01-26| 315.2|concluido| Lucas Almeida|    RS|
|       104|      16|2023-01-30| 210.0|concluido|       Janaina|    PR|
+----------+--------+----------+------+---------+--------------+------+



## UPDATE — Atualizando valor da venda

In [8]:
spark.read.format("delta").load("../data/delta/vendas_sul").createOrReplaceTempView("vendas_delta")
spark.sql("UPDATE vendas_delta SET valor = valor * 1.10 WHERE id_venda = 1")
print('UPDATE realizado!')
spark.sql("SELECT * FROM vendas_delta").show()

UPDATE realizado!
+----------+--------+----------+------+---------+--------------+------+
|id_cliente|id_venda|data_venda| valor|   status|          nome|estado|
+----------+--------+----------+------+---------+--------------+------+
|       101|       1|2023-01-15|275.55|concluido|     Ana Julia|    SC|
|       102|       2|2023-01-16| 120.0|concluido|Laura Silveira|    RS|
|       101|       4|2023-01-18|  50.0|concluido|     Ana Julia|    SC|
|       104|       5|2023-01-19|1500.0|concluido|       Janaina|    PR|
|       110|      11|2023-01-25| 600.0|concluido|  Camila Rocha|    SC|
|       111|      12|2023-01-26| 315.2|concluido| Lucas Almeida|    RS|
|       104|      16|2023-01-30| 210.0|concluido|       Janaina|    PR|
+----------+--------+----------+------+---------+--------------+------+



## DELETE — Removendo uma venda

In [9]:
spark.sql("DELETE FROM vendas_delta WHERE id_venda = 4")
print('DELETE realizado!')
spark.sql("SELECT * FROM vendas_delta").show()

DELETE realizado!
+----------+--------+----------+------+---------+--------------+------+
|id_cliente|id_venda|data_venda| valor|   status|          nome|estado|
+----------+--------+----------+------+---------+--------------+------+
|       101|       1|2023-01-15|275.55|concluido|     Ana Julia|    SC|
|       102|       2|2023-01-16| 120.0|concluido|Laura Silveira|    RS|
|       104|       5|2023-01-19|1500.0|concluido|       Janaina|    PR|
|       110|      11|2023-01-25| 600.0|concluido|  Camila Rocha|    SC|
|       111|      12|2023-01-26| 315.2|concluido| Lucas Almeida|    RS|
|       104|      16|2023-01-30| 210.0|concluido|       Janaina|    PR|
+----------+--------+----------+------+---------+--------------+------+

